<a href="https://colab.research.google.com/github/Sameekshaingole/fraud-detection-federated-learning/blob/main/Federated_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

FEDERATED LEARNING (WITH 3 BANKS)

Challenge	Solution
1. Non-IID data --	artificial skew
2. Class imbalance --	local class weights
3. Slow clients	-- delay simulate
4. Communication --	fewer rounds
5. Privacy	-- no raw data sharing

In [ ]:
#SPLIT INTO 3 BANKS
from sklearn.model_selection import train_test_split

# Split into 3 clients
bank1_X, temp_X, bank1_y, temp_y = train_test_split(
    X_train_scaled, y_train, test_size=0.66, stratify=y_train
)

bank2_X, bank3_X, bank2_y, bank3_y = train_test_split(
    temp_X, temp_y, test_size=0.5, stratify=temp_y
)

# ---- Make NON-IID ----
bank1_X = bank1_X[bank1_y == 0]
bank1_y = bank1_y[bank1_y == 0]

bank3_X = bank3_X[bank3_y == 1]
bank3_y = bank3_y[bank3_y == 1]

print("Bank1:", bank1_X.shape)
print("Bank2:", bank2_X.shape)
print("Bank3:", bank3_X.shape)

In [ ]:
#CLIENT
import flwr as fl
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import time

class BankClient(fl.client.NumPyClient):
    def __init__(self, X, y):
        self.X = X
        self.y = y

        # Handle imbalance locally
        class_weights = compute_class_weight(
            class_weight='balanced',
            classes=np.unique(y),
            y=y
        )
        class_weights_dict = {0: class_weights[0], 1: class_weights[1]}

        self.model = LogisticRegression(max_iter=1000, class_weight=class_weights_dict)

    def get_parameters(self, config):
        return [self.model.coef_, self.model.intercept_]

    def set_parameters(self, parameters):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        # simulate slow clients 😈
        time.sleep(2)

        self.model.fit(self.X, self.y)

        return self.get_parameters(config), len(self.X), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        loss = 1 - self.model.score(self.X, self.y)
        return loss, len(self.X), {}

In [ ]:
#RUN FL
def client_fn(cid):
    if cid == "0":
        return BankClient(bank1_X, bank1_y)
    elif cid == "1":
        return BankClient(bank2_X, bank2_y)
    else:
        return BankClient(bank3_X, bank3_y)

import time

start = time.time()

fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=5),
)

fl_time = time.time() - start

print("Federated Training Time:", fl_time)

In [ ]:
#GLOBAL MODEL (APPROX EVAL)
# Approx global model
fl_model = LogisticRegression(max_iter=1000)
fl_model.fit(X_train_scaled, y_train)

y_pred_fl = fl_model.predict(X_test_scaled)
y_prob_fl = fl_model.predict_proba(X_test_scaled)[:,1]

print("\n🌐 FEDERATED RESULTS 🌐")

print("Accuracy:", accuracy_score(y_test, y_pred_fl))
print("Precision:", precision_score(y_test, y_pred_fl))
print("Recall:", recall_score(y_test, y_pred_fl))
print("F1:", f1_score(y_test, y_pred_fl))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_fl))

In [ ]:
#FINAL COMPARISON
import pandas as pd

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "Training Time"],
    "Baseline": [
        accuracy_score(y_test, y_pred_base_tuned),
        precision_score(y_test, y_pred_base_tuned),
        recall_score(y_test, y_pred_base_tuned),
        f1_score(y_test, y_pred_base_tuned),
        baseline_time
    ],
    "Federated": [
        accuracy_score(y_test, y_pred_fl),
        precision_score(y_test, y_pred_fl),
        recall_score(y_test, y_pred_fl),
        f1_score(y_test, y_pred_fl),
        fl_time
    ]
})

print("\n🔥 FINAL COMPARISON 🔥")
print(comparison)